In [1]:
import os
import csv
import datetime as dt
import pandas as pd

# index를 날짜로 하는 데이터프레임을 price, adj_price, divend에 대해 반환
def extract_stock_data(file_path, ticker):
    extract_datas = []

    with open(os.path.join(file_path, ticker + '.csv'), newline='') as csvfile:
        reader = csv.reader(csvfile)

        for row in reader:
            extract_datas.append(row)
        
    price_dates = list(map(dt.date.fromisoformat, extract_datas[0]))
    price_history = list(map(float, extract_datas[1]))
    adj_price_dates = list(map(dt.date.fromisoformat, extract_datas[2]))
    adj_price_history = list(map(float, extract_datas[3]))
    divend_dates = list(map(dt.date.fromisoformat, extract_datas[4]))
    divend_history = list(map(float, extract_datas[5]))

    price_df = pd.DataFrame({'date': price_dates, 'price': price_history})
    adj_price_df = pd.DataFrame({'date': adj_price_dates, 'adj_price': adj_price_history})
    divend_df = pd.DataFrame({'date': divend_dates, 'divend': divend_history})

    price_df.set_index('date', inplace=True)
    adj_price_df.set_index('date', inplace=True)
    divend_df.set_index('date', inplace=True)

    return (price_df, adj_price_df, divend_df)

def make_stock_data(file_path, tickers):
    stock_data = []

    for ticker in tickers:
        price_df, adj_price_df, divend_df = extract_stock_data(file_path, ticker)
        stock_df = pd.concat([price_df, adj_price_df, divend_df], axis=1, sort=True)
        stock_df.columns = pd.MultiIndex.from_product([[ticker], stock_df.columns])
        stock_data.append(stock_df)

    stock_data = pd.concat(stock_data, axis=1, sort=True)

    return stock_data


In [2]:
from portfolio import Portfolio

# stock 불러와서 데이터프레임화 하기
file_path = './etf'
tickers = ['QQQ', 'DGRW', 'SCHD', 'SPY', 'SCHG', 'SPYG']
stock_data = make_stock_data(file_path, tickers)
pd.options.display.float_format = '{:.2f}'.format

# test_stock = Portfolio(make_stock_data(file_path, ['QQQ', 'SCHG']), 10000)
test_stock = Portfolio(stock_data, 10000)
portfolio = Portfolio(stock_data, 10000, {"QQQ":56, "SCHD":24, "DGRW":20})

In [3]:
def calc_target_ratio(base_ratio:tuple, etc_ratio:tuple) -> tuple:
    base_sum = sum(base_ratio)
    qqq = base_ratio[0] / base_sum * 100
    etc_sum = sum(etc_ratio)
    etc1 = (etc_ratio[0] / etc_sum) * (base_ratio[1] / base_sum) * 100
    etc2 = (etc_ratio[1] / etc_sum) * (base_ratio[1] / base_sum) * 100

    return (round(qqq, 1), round(etc1, 1), round(etc2, 1))

In [ ]:
from joblib import Parallel, delayed
from portfolio_test import portfolio_backtest_by_duration

stats = pd.DataFrame(columns=['cagr', 'stdev', 'mdd', 'beta', 'alpha'])
benchmark = stock_data[('QQQ', 'adj_price')]

param_list = []
for qqq_weight in range(0, 100, 5):
    for schd_weight in range(0, 10):
        ratio_qqq_etc = (qqq_weight, 100 - qqq_weight)
        ratio_schd_dgrw = (schd_weight, 10 - schd_weight)
        ratio_qqq_schd_dgrw = calc_target_ratio(ratio_qqq_etc, ratio_schd_dgrw)
        
        target_ratio = {
            'QQQ': ratio_qqq_schd_dgrw[0],
            'SCHD': ratio_qqq_schd_dgrw[1],
            'DGRW': ratio_qqq_schd_dgrw[2]
        }
        print(target_ratio)    
        p = Portfolio(stock_data, 10000, target_ratio)
        param_list.append((p, benchmark, 5))

# with Parallel(n_jobs=-1) as parallel:
#         results = parallel(delayed(portfolio_backtest_by_duration)(*params) for params in param_list)
        
# for stat in results:
#     stats = pd.concat([stats, stat])

# stats

In [35]:
stats.to_csv('./stats_qqq_schd_dgrw.csv')

In [61]:
stats = pd.read_csv('./stats_qqq_schd_dgrw.csv', index_col=0)
stats

In [74]:
weight_map = {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0,
              5: 1.1, 6: 1.1,
              7: 1.2, 8: 1.2}

stats['weight'] = stats.index.map(lambda x: weight_map[x % 9])
stats

def weighted_alpha_mean(group):
    d = group['alpha']
    w = group['weight']
    return (d * w).sum() / w.sum()

weighted_alpha_stat = stats.groupby('ratio').apply(weighted_alpha_mean)
weighted_alpha_stat = weighted_alpha_stat.sort_values(ascending=False)

pd.set_option('display.max_rows', None)
weighted_alpha_stat

In [ ]:
stat_summary = stats.groupby(['ratio']).mean()
stat_summary['alpha'] = weighted_alpha_stat
stat_summary = stat_summary.sort_values(by='alpha', ascending=False)
stat_summary = stat_summary[stat_summary['alpha'] > 0]
stat_summary.sort_values(by='mdd', ascending=True, inplace=True)
stat_summary[(stat_summary['cagr'] > 14.5) & (stat_summary['alpha'] > 0.15)]

,cagr,stdev,mdd,beta,alpha,weight
ratio,,,,,,
35.0:39.0:26.0,14.52,17.32,25.43,0.74,0.19,1.07
40.0:48.0:12.0,14.78,17.53,25.56,0.75,0.25,1.07
40.0:54.0:6.0,14.71,17.52,25.58,0.75,0.26,1.07
40.0:36.0:24.0,14.83,17.58,25.59,0.76,0.16,1.07
40.0:42.0:18.0,14.82,17.56,25.59,0.76,0.21,1.07
45.0:49.5:5.5,15.07,17.79,25.76,0.77,0.24,1.07
45.0:44.0:11.0,15.07,17.80,25.79,0.77,0.18,1.07
50.0:35.0:15.0,15.52,18.13,25.90,0.80,0.19,1.07
50.0:40.0:10.0,15.44,18.09,25.95,0.79,0.18,1.07


In [41]:
pd.set_option('display.max_rows', None)

stat_summary_sorted = stat_summary.sort_values(by=['alpha'], ascending=False)
stat_summary_sorted = stat_summary_sorted[stat_summary_sorted['alpha'] > 0]
stat_sorted = stats.sort_values(by=['ratio', 'alpha'], ascending=[True, False])

In [ ]:
import seaborn as sns

sns.catplot(data=stats, x='ratio', y='cagr', kind='box')
sns.catplot(data=stats, x='ratio', y='stdev', kind='box')
sns.catplot(data=stats, x='ratio', y='mdd', kind='box')

In [ ]:
sns.relplot(data=test, x='stdev', y='cagr', hue='ratio')

In [ ]:
sns.relplot(data=stats, x='stdev', y='cagr', hue='ratio')